In [ ]:
!pip install -q sentence-transformers transformers accelerate gradio

In [ ]:
from google.colab import files
uploaded = files.upload()
name = list(uploaded.keys())[0]
text = uploaded[name].decode("utf-8-sig")
print(name, len(text))

In [ ]:
import re, torch
from sentence_transformers import SentenceTransformer, util

def make_chunks(text, max_chars=600, overlap_sents=1):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', text) if s.strip()]
    chunks, cur = [], []
    for s in sents:
        if cur and sum(len(x) for x in cur) + len(s) > max_chars:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sents:]
        cur.append(s)
    if cur:
        chunks.append(" ".join(cur))
    return chunks

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedder.max_seq_length = 256

chunks = make_chunks(text)
vectors = embedder.encode(chunks, convert_to_tensor=True,
                          normalize_embeddings=True, show_progress_bar=True)
print(len(chunks), "чанков")

def retrieve(query, k=4):
    q = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(q, vectors)[0]
    top = scores.topk(min(k, len(chunks)))
    return [(float(s), int(i)) for s, i in zip(top.values, top.indices)]

for s, i in retrieve("Что такое условная вероятность?", k=2):
    print(round(s, 2), chunks[i][:150], "\n")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
tok = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")

In [ ]:
SYSTEM = ("Ты помощник, который отвечает на вопросы строго по приведённому контексту. "
          "Если в контексте нет ответа, скажи: «В тексте этого нет». "
          "Отвечай по-русски, кратко и в конце укажи номера использованных фрагментов в квадратных скобках.")

def answer(question, k=4):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{n}] {chunks[i]}" for n, (s, i) in enumerate(hits, 1))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=300, do_sample=False)
    ans = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return ans, hits

ans, hits = answer("Почему распределение суммы большого числа независимых величин близко к нормальному?")
print(ans)

In [ ]:
eval_set = [
    {"q": "Чем числовая характеристика результата случайного эксперимента отличается от обычной величины?", "key": "Случайные величины"},
    {"q": "Как называется средний уровень случайной величины в вероятностном смысле?", "key": "Математическое ожидание"},
    {"q": "Какими показателями измеряют разброс значений относительно среднего уровня?", "key": "Дисперсия и стандартное отклонение"},
    {"q": "Почему распределение суммы большого числа независимых величин часто оказывается близким к нормальному?", "key": "Центральная предельная теорема"},
    {"q": "Почему модель может хорошо работать на знакомых данных, но ошибаться на новых?", "key": "Переобучение и обобщающая способность"},
]

In [ ]:
import numpy as np

def evaluate(eval_set, k=5):
    hits_n, rr = 0, []
    for item in eval_set:
        rank = None
        for r, (s, i) in enumerate(retrieve(item["q"], k), start=1):
            if item["key"].lower() in chunks[i].lower():
                rank = r
                break
        hits_n += rank is not None
        rr.append(1 / rank if rank else 0)
    return {"recall@5": hits_n / len(eval_set), "MRR": float(np.mean(rr))}

print(evaluate(eval_set))

In [ ]:
import gradio as gr

def ui(q):
    ans, hits = answer(q)
    src = "\n\n".join(f"[{n}] (сходство {s:.2f})\n{chunks[i]}"
                      for n, (s, i) in enumerate(hits, 1))
    return ans, src

demo = gr.Interface(
    fn=ui,
    inputs=gr.Textbox(label="Вопрос"),
    outputs=[gr.Textbox(label="Ответ"), gr.Textbox(label="Найденные фрагменты", lines=12)],
    title="RAG по тексту",
)
demo.launch(share=True)